In [ ]:
%run common.py
import sys
sys.path.append("../../legal-data-clustering/")
%run '../../legal-data-clustering/legal_data_clustering/utils/graph_api.py'
import multiprocessing
from datetime import datetime
import glob
from collections import defaultdict
from matplotlib import rc
import matplotlib
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_squared_error, mean_squared_log_error
import matplotlib.pyplot as plt
plt.style.use('altair.mplstyle')

# Load graph

In [ ]:
G = nx.read_gpickle('../../legal-networks-data/de_decisions/2_network.gpickle.gz')

In [ ]:
G_md = quotient_decision_graph(G, merge_decisions=True, merge_statutes=False)
print('G_md done')
# G_ms = quotient_decision_graph(G, merge_decisions=False, merge_statutes=True)
print('G_ms done')
G_md_ms = quotient_decision_graph(G, merge_decisions=True, merge_statutes=True)

# Loading Data

In [ ]:
decision_reference_count = defaultdict(int)
for decision, b in G_md_ms.nodes(data='bipartite'):
    if b == 'decision':
        for u, v in G_md_ms.out_edges(decision):
            decision_reference_count[u] += G_md_ms.edges[u, v]['weight']

In [ ]:
decision_reference_binary_statute_count = dict(
    G_md_ms.out_degree([
        n 
        for n, b in G_md_ms.nodes(data='bipartite') 
        if b == 'decision'
    ])
)
decision_reference_binary_norm_count = dict(
    G_md.out_degree([
        n 
        for n, b in G_md.nodes(data='bipartite') 
        if b == 'decision'
    ])
)

In [ ]:
node_docs = [
    n 
    for n, b in G_md_ms.nodes(data='bipartite') 
    if b == 'decision'
]
seqitems_count_dict = {n.split('_')[0]: 0 for n in node_docs}
for n, t in G.nodes(data='type'):
    if t == 'seqitem':
        seqitems_count_dict[n.split('_')[0]] += 1

In [ ]:
raw = list(dict(decision_reference_count).items())
df_decision_ref = pd.DataFrame(index=list(decision_reference_count))
df_decision_ref['Referenzen (Gewichtet)'] = [
    decision_reference_count[n] for n in df_decision_ref.index
]
df_decision_ref['Referenzen (Binär: §/Art)'] = [
    decision_reference_binary_norm_count[n] for n in df_decision_ref.index
]
df_decision_ref['Referenzen (Binär: Gesetz)'] = [
    decision_reference_binary_statute_count[n] for n in df_decision_ref.index
]
df_decision_ref['Jahr'] = [
    G_md_ms.nodes[n]['datum'][:4] for n in df_decision_ref.index
]
df_decision_ref['Gericht'] = [
    G_md_ms.nodes[n]['gericht'] for n in df_decision_ref.index
]
df_decision_ref['Spruchkörper'] = [
    G_md_ms.nodes[n]['spruchkoerper'] for n in df_decision_ref.index
]
df_decision_ref['Token'] = [
    G_md_ms.nodes[n]['tokens_n'] for n in df_decision_ref.index
]
df_decision_ref['Absätze'] = [
    seqitems_count_dict[n] for n in df_decision_ref.index
]

In [ ]:
df_decision_ref.Absätze = [
    max(1, v) for v in df_decision_ref.Absätze
]

# Anzahl

In [ ]:
diss_data(
    'makro_decisions_refs_count',
    de_num_format(f"{df_decision_ref['Referenzen (Gewichtet)'].sum():,}"),
)
diss_data(
    'makro_decisions_refs_count_binary_art',
    de_num_format(f"{df_decision_ref['Referenzen (Binär: §/Art)'].sum():,}"),
)
diss_data(
    'makro_decisions_refs_count_binary_gesetz',
    de_num_format(f"{df_decision_ref['Referenzen (Binär: Gesetz)'].sum():,}"),
)

In [ ]:
df_decision_ref[df_decision_ref.Token > 3000]

In [ ]:
charts = []
for field in ['Referenzen (Gewichtet)', 'Referenzen (Binär: §/Art)', 'Referenzen (Binär: Gesetz)']:
    last = field == 'Referenzen (Binär: Gesetz)'
    df = df_decision_ref[df_decision_ref.Jahr<='2020'].groupby('Jahr').sum().reset_index()
#     df = df_decision_ref.groupby(['Jahr', 'Gericht']).sum().reset_index()
    charts.append(alt.Chart(
        df[df.Jahr <= '2020']
    ).mark_line(point=alt.OverlayMarkDef(size=25), size=2).encode(
        (
            alt.X('Jahr:O') 
            if field == 'Referenzen (Binär: Gesetz)' 
            else alt.X('Jahr:O', title=None, axis=alt.Axis(labels=False, ticks=False))
        ),
        alt.Y(
            f'{field}:Q', 
            scale=alt.Scale(zero=False),
            axis=alt.Axis(tickCount=3),
            title=(
                field.replace('(','').replace(')','').split(maxsplit=1)
            )
        ),
#         alt.Color('Gericht:N')
    ).properties(height=60, width=300))
chart = alt.vconcat(*charts).configure_concat(
    spacing=5
)
save_chart(chart, 'makro_decisions_reference_count')

In [ ]:
chart_graycolor = chart.configure_line(color="black").configure_point(color="black")
save_chart(chart_graycolor, 'makro_decisions_reference_count_graycolor')

# Dichte

## Pro Entscheidung

In [ ]:
def reference_density_boxplot(df_decision_ref, col, col_title, colors):
    global df
    df = df_decision_ref[(df_decision_ref.Gericht!='GmSOGB')].groupby(['Jahr', 'Gericht']).describe(percentiles=[.05, .25, .5, .75, .95])[col].reset_index()
    df_all = df_decision_ref.groupby('Jahr').describe(percentiles=[.05, .25, .5, .75, .95])[col].reset_index()
    df_all['Gericht'] = 'Alle'
    df = df.append(df_all, sort=True)

    chart = alt.LayerChart(df).encode(
        x='Jahr:O',
        tooltip=['min:Q', '5%:Q', '25%:Q', '50%:Q', '75%:Q', '95%:Q', 'max:Q'],
    ).add_layers(
        alt.Chart().mark_rule().encode(y=alt.Y('5%:Q', title=col_title), y2='95%:Q'),
        alt.Chart().mark_rule().encode(y=alt.Y('25%:Q'), y2='75%:Q'),
        alt.Chart().mark_bar(width=10).encode(
            y='25%:Q', 
            y2='75%:Q', 
            color=alt.Color(
                'Gericht:N', 
                legend=None,
                scale=alt.Scale(range=colors)
            )
        ),
        alt.Chart().mark_tick(thickness=2, color='white', width=10).encode(y='50%:Q'),
    #     alt.Chart().mark_tick(color='red', width=15).encode(y='mean:Q'),


    ).properties(
        width=140,
        height=200
    ).facet(
        facet='Gericht:N',
        columns=4,
        spacing=5
    )
    return chart

In [ ]:
chart = reference_density_boxplot(df_decision_ref[
#     (df_decision_ref.Token >= 3000)&
    (df_decision_ref.Jahr <= '2020')
], 'Referenzen (Gewichtet)', ['Referenzen', 'einer Gerichtsentscheidung'], alle_gericht_scale_range)
chart = large_labels(chart)
save_chart(chart, 'makro_decisions_referenz_boxplot')

In [ ]:
chart = reference_density_boxplot(df_decision_ref[
#     (df_decision_ref.Token >= 3000)&
    (df_decision_ref.Jahr <= '2020')
], 'Referenzen (Gewichtet)', ['Referenzen', 'einer Gerichtsentscheidung'], ["black"])
chart = large_labels(chart)
save_chart(chart, 'makro_decisions_referenz_boxplot_graycolor')

In [ ]:
min_size = 3000

dfs = []

for lang, df_filtered in (
    (
        'Alle', 
        df_decision_ref
    ),
    (
        f'Mindestens {min_size} Token', 
        df_decision_ref[df_decision_ref.Token >= min_size]
    )
):
    stats_df = df_filtered.groupby(['Jahr', 'Gericht']).mean().reset_index()
    stats_df['lang'] = lang
    dfs.append(stats_df)
    
    stats_df_all = df_filtered.groupby('Jahr').mean().reset_index()
    stats_df_all['Gericht'] = 'Alle'
    stats_df_all['lang'] = lang
    dfs.append(stats_df_all)
    
df = pd.concat(dfs, sort=False)

df = df[
    (df.Gericht != 'GmSOGB')&
    (df.Jahr <= '2020')
]

rows = [
    'Referenzen (Gewichtet)', 'Referenzen (Binär: §/Art)', 'Referenzen (Binär: Gesetz)'
]
chart = alt.Chart(df).mark_line(point=alt.OverlayMarkDef(size=40)).transform_fold(rows).encode(
    alt.X('Jahr:O'),
    alt.Y(f'value:Q', title=None),
    alt.Color(
        'Gericht:N', 
        scale=alt.Scale(range=alle_gericht_scale_range), 
        legend=None,
    ),
    alt.Row('lang:N', title=None),
    alt.Column('key:N', title='Durchschnitt: Anzahl pro Entscheidung', sort=rows, header=alt.Header(titlePadding=0)),
    alt.Shape('Gericht:N', legend=None, scale=alt.Scale(range=['circle','square',"diamond",'triangle-up','triangle-down','triangle-left','triangle-right'])),
).resolve_scale(y='independent').properties(
    width=140,
    height=190
).configure_facet(spacing=5)
save_chart(chart, 'makro_decisions_reference_gesetz_mean')

In [ ]:
legend_chart = alt.Chart(df).mark_point(filled=True, opacity=1, size=0).transform_fold(rows).encode(
    alt.Color('Gericht:N', scale=alt.Scale(range=alle_gericht_scale_range)),
    alt.Shape('Gericht:N', legend=alt.Legend(orient='bottom', direction='horizontal', offset=5), scale=alt.Scale(range=['circle', 'square',"diamond",'triangle-up','triangle-down','triangle-left','triangle-right']))
).configure_view(strokeWidth=0).properties(width=1, height=1, padding={'right': 30})
save_chart(legend_chart, f'makro_decisions_reference_gesetz_mean_legend')

In [ ]:
chart_graycolor = chart.encode(
    alt.Color(
        'Gericht:N', 
        legend=None,
        scale=alt.Scale(range=[ '#000', '#787878', '#9a9a9a', '#BBB']),
    ),
)
save_chart(chart_graycolor, f'makro_decisions_reference_gesetz_mean_graycolor')

In [ ]:
legend_chart_graycolor = legend_chart.encode(
    alt.Color(
        'Gericht:N', 
        scale=alt.Scale(range=[ '#000', '#787878', '#9a9a9a', '#BBB']),
    ),
)
save_chart(legend_chart_graycolor, f'makro_decisions_reference_gesetz_mean_legend_graycolor')

## Pro Token

In [ ]:
df_decision_ref_rel_token = df_decision_ref.copy()
df_decision_ref_rel_token['Referenzen (Gewichtet)'] = df_decision_ref_rel_token.Token / \
    df_decision_ref_rel_token['Referenzen (Gewichtet)']
df_decision_ref_rel_token['Referenzen (Binär: Gesetz)'] = df_decision_ref_rel_token.Token / \
    df_decision_ref_rel_token['Referenzen (Binär: Gesetz)'] 
df_decision_ref_rel_token['Referenzen (Binär: §/Art)'] = df_decision_ref_rel_token.Token / \
    df_decision_ref_rel_token['Referenzen (Binär: §/Art)'] 

In [ ]:
min_size = 3000

dfs = []

for lang, df_filtered in (
    (
        'Alle', 
        df_decision_ref_rel_token
    ),
    (
        f'Mindestens {min_size} Token', 
        df_decision_ref_rel_token[df_decision_ref_rel_token.Token >= min_size]
    )
):
    stats_df = df_filtered.groupby(['Jahr', 'Gericht']).mean().reset_index()
    stats_df['lang'] = lang
    dfs.append(stats_df)
    
    stats_df_all = df_filtered.groupby('Jahr').mean().reset_index()
    stats_df_all['Gericht'] = 'Alle'
    stats_df_all['lang'] = lang
    dfs.append(stats_df_all)
    
df = pd.concat(dfs, sort=False)

df_chart = df[
    (df.Gericht != 'GmSOGB')&
    (df.Gericht != 'BPatG')&
    (df.Jahr <= '2020')
]

rows = [
    'Referenzen (Gewichtet)', 'Referenzen (Binär: §/Art)', 'Referenzen (Binär: Gesetz)'
]
chart = alt.Chart(df_chart).mark_line(point=alt.OverlayMarkDef(size=40)).transform_fold(rows).encode(
    alt.X('Jahr:O'),
    alt.Y(
        f'Referenzen (Gewichtet):Q', 
        title=['Durchschnitt: Tokens/Referenzen', 'pro Entscheidung'],
    ),
    alt.Color('Gericht:N', scale=alt.Scale(range=alle_gericht_ohne_bpatg_scale_range), legend=None),
    alt.Column('lang:N', title=None),
#     alt.Row('key:N', title='Durchschnitt: Anzahl pro Entscheidung', sort=rows)
    alt.Shape('Gericht:N', legend=None, scale=alt.Scale(range=['circle','square',"diamond",'triangle-up','triangle-down','triangle-left','triangle-right'])),
).properties(
    width=150,
    height=150
)
save_chart(chart, 'makro_decisions_reference_token_mean')

In [ ]:
legend_chart = alt.Chart(df_chart).mark_point(filled=True, opacity=1, size=0).encode(
    alt.Color('Gericht:N', scale=alt.Scale(range=alle_gericht_ohne_bpatg_scale_range)),
    alt.Shape('Gericht:N', scale=alt.Scale(range=['circle', 'square',"diamond",'triangle-up','triangle-down','triangle-left','triangle-right']))
).configure_view(strokeWidth=0).properties(width=1, height=1, padding={'top': 28}).configure_legend(
    offset=13,  # Adjust distance
)
save_chart(legend_chart, f'makro_decisions_reference_token_mean_legend')

In [ ]:
chart_graycolor = chart.encode(
    alt.Color(
        'Gericht:N', 
        legend=None,
        scale=alt.Scale(range=[ '#000', '#787878', '#9a9a9a', '#BBB']),
    ),
)
save_chart(chart_graycolor, f'makro_decisions_reference_token_mean_graycolor')

In [ ]:
legend_chart_graycolor = legend_chart.encode(
    alt.Color(
        'Gericht:N', 
        scale=alt.Scale(range=[ '#000', '#787878', '#9a9a9a', '#BBB']),
    ),
)
save_chart(legend_chart_graycolor, f'makro_decisions_reference_token_mean_legend_graycolor')

In [ ]:
bpatg_limits_df = df[
    (df.Jahr <= '2020')&
    (df.Gericht == 'BPatG')
].groupby('lang')['Referenzen (Gewichtet)'].describe()

min_alle = de_num_format(f"{bpatg_limits_df.loc['Alle', 'min']:,.2f}")
max_alle = de_num_format(f"{bpatg_limits_df.loc['Alle', 'max']:,.2f}")
min_lang = de_num_format(f"{bpatg_limits_df.loc[f'Mindestens {min_size} Token', 'min']:,.2f}")
max_lang = de_num_format(f"{bpatg_limits_df.loc[f'Mindestens {min_size} Token', 'max']:,.2f}")

diss_data(
    'makro_decisions_reference_token_mean_bpatg_limits', 
    f'{min_alle} und {max_alle} bzw. {min_lang} und {max_lang}'
)

In [ ]:
df = df_decision_ref_rel_token[df_decision_ref_rel_token.Token >= min_size].groupby(['Jahr', 'Gericht']).mean()\
    .reset_index()
df = df[(df.Gericht != 'GmSOGB') & (df.Gericht != 'BPatG')]
df = df[
    (df.Gericht != 'GmSOGB')&
    (df.Gericht != 'BPatG')&
    (df.Jahr <= '2020')
]
chart = alt.Chart(df).mark_line(point=alt.OverlayMarkDef(size=25)).encode(
    alt.X('Jahr:O'),
    alt.Y('Referenzen (Gewichtet):Q', title='Tokens/Referenzen'),
    alt.Color('Gericht:N'),
)
chart
# save_chart(chart, 'makro_decisions_reference_token_mean')

## Pro Absatz

In [ ]:
df_decision_ref_rel_seqitem = df_decision_ref.copy()
df_decision_ref_rel_seqitem['Referenzen (Gewichtet)'] /= df_decision_ref_rel_seqitem.Absätze
df_decision_ref_rel_seqitem['Referenzen (Binär: Gesetz)'] /= df_decision_ref_rel_seqitem.Absätze
df_decision_ref_rel_seqitem['Referenzen (Binär: §/Art)'] /= df_decision_ref_rel_seqitem.Absätze

In [ ]:
df = df_decision_ref_rel_seqitem.groupby(['Jahr', 'Gericht']).mean()\
    .reset_index()
df = df[(df.Gericht != 'GmSOGB')]

chart = alt.Chart(df).mark_line(point=alt.OverlayMarkDef(size=25)).encode(
    alt.X('Jahr:O'),
    alt.Y('Referenzen (Gewichtet):Q', title='Referenzen/Absätze'),
    alt.Color('Gericht:N'),
)
chart
save_chart(chart, 'makro_decisions_reference_seqitem_mean')

## Korrelation Referenzen/Absatz

In [ ]:
gerichte_list = [None, 'BAG', 'BFH', 'BGH', 'BPatG', 'BSG', 'BVerfG', 'BVerwG']

In [ ]:
df_decision_ref

In [ ]:
def plot_linreg(x, y, ax=None, title=None, stats_df=None, xlim=None, ylim=None, xjitter=False, 
                show_x_label=True, show_y_label=True, ylabel=None, color=True):
    plt.style.use('altair.mplstyle')
    x_jittered = jitter(x) if xjitter else x
    
    if ax is None:
        _, ax = plt.subplots()
    
    reg = LinearRegression().fit(
        x.to_numpy().reshape(-1, 1), 
        y
    )
    y_predicted = [reg.predict([[x]]) for x in x]
    if stats_df is not None:
        stats_df.loc[len(stats_df)]= dict(
            Gericht=title or 'Alle',
            a=reg.intercept_,
            b=reg.coef_[0],
            r2=r2_score(y, y_predicted),
            mse=mean_squared_error(y, y_predicted),
            msle=mean_squared_log_error(y, y_predicted),
        )

    ax.title.set_text((title or 'Alle') + f' (r={r2_score(y, y_predicted):.3f})')
    ax.spines['bottom'].set_edgecolor('#888888')
    ax.spines['left'].set_edgecolor('#888888')
    ax.scatter(x_jittered, y, s=5, alpha=.05, color='k')
    if show_x_label:
        ax.set_xlabel(x.name)
    if show_y_label:
        ax.set_ylabel(ylabel or y.name)
    ax.tick_params(labelleft=show_y_label)
    ax.tick_params(labelbottom=show_x_label)
    ax.set_ylim(0, ylim or y.quantile(.999))
    _, x_max = ax.set_xlim(0, xlim or max(int(x.quantile(.999)), 70))
    ax.yaxis.set_major_formatter(matplotlib.ticker.StrMethodFormatter('{x:,.0f}'))
#     plt.xticks(rotation=90)
#     ax.set_xticklabels(ax.get_xticklabels(), rotation=90)
    if len(x) >= 5:
        line_x = [0, x_max]
        line_y = [reg.predict([[x]]) for x in line_x]
        if color:
            ax.plot(line_x, line_y, color='red')
        else:
            ax.plot(line_x, line_y, color='white')
            ax.plot(line_x, line_y, color='k', linestyle='dotted')
    ax.get_yaxis().set_major_formatter(matplotlib.ticker.FuncFormatter(lambda x, p: format(x, ',.0f').replace(',', '.')))

In [ ]:
for color in [True, False]:
    rc('axes', titlepad=5.0)
    fig, axs = plt.subplots(nrows=3, ncols=3, figsize=(8,8))
    stats_df = pd.DataFrame(columns=['Gericht', 'a', 'b', 'r2', 'mse', 'msle'])
    for idx, (gericht, ax) in enumerate(zip(
        gerichte_list,
        itertools.chain.from_iterable(axs)
    )):
        df = df_decision_ref[df_decision_ref.Gericht == gericht] if gericht else df_decision_ref
        plot_linreg( 
            df['Token'], 
            df['Absätze'], 
            title=gericht, 
            ax=ax,
            stats_df=stats_df,
            xlim=17500,
            ylim=200,
            show_x_label=bool(idx > 5),
            show_y_label=bool(idx % 3 == 0),
            color=color
        )
    axs[-1, -1].axis('off')
    plt.subplots_adjust(top=0.97, bottom=0.07, left=0.09, right=0.99, wspace=0.1)
    fig.savefig(f'{data_figures_path}/makro_decisions_seqitems_tokens_scatter{"" if color else "_graycolor"}.png', dpi=300) 
    stats_df = stats_df.set_index('Gericht')

In [ ]:
diss_data('makro_decisions_seqitems_tokens_r2_bverfg', de_num_format(f"{stats_df.loc['BVerfG', 'r2']:.3f}"))
diss_data('makro_decisions_seqitems_tokens_r2_bpatg', de_num_format(f"{stats_df.loc['BPatG', 'r2']:.3f}"))

pd.options.display.float_format = '{:.4}'.format

stats_df.sort_values('r2')

In [ ]:
for color in [True, False]:
    rc('axes', titlepad=5.0)
    fig, axs = plt.subplots(nrows=3, ncols=3, figsize=(8,8))
    stats_df = pd.DataFrame(columns=['Gericht', 'a', 'b', 'r2', 'mse', 'msle'])
    for idx, (gericht, ax) in enumerate(zip(
        gerichte_list,
        itertools.chain.from_iterable(axs)
    )):
        df = df_decision_ref[df_decision_ref.Gericht == gericht] if gericht else df_decision_ref
        plot_linreg( 
            df['Referenzen (Gewichtet)'], 
            df['Token'], 
            title=gericht, 
            ax=ax,
            stats_df=stats_df,
            xlim=600,
            ylim=17500,
            show_x_label=bool(idx > 5),
            show_y_label=bool(idx % 3 == 0),
            ylabel='Tokens',
            color=color
        )
    axs[-1, -1].axis('off')
    plt.subplots_adjust(top=0.97, bottom=0.07, left=0.09, right=0.99, wspace = 0.1)
    fig.savefig(f'{data_figures_path}/makro_decisions_reference_token_scatter{"" if color else "_graycolor"}.png', dpi=300)

In [ ]:
stats_df[
    (stats_df.Gericht != 'BPatG')&
    (stats_df.Gericht != 'GmSOGB')
].r2.min()

In [ ]:
r2_min = stats_df[
    (stats_df.Gericht != 'BPatG')&
    (stats_df.Gericht != 'GmSOGB')
].r2.min()

r2_max = stats_df[
    (stats_df.Gericht != 'BPatG')&
    (stats_df.Gericht != 'GmSOGB')
].r2.max()

diss_data(
    'makro_decisions_reference_token_scatter_rho_min_max',
    de_num_format(f'{r2_min:,.2f}') + 
    ' und ' + 
    de_num_format(f'{r2_max:,.2f}')
)

pd.options.display.float_format = '{:,.2f}'.format

stats_df.sort_values('r2')

In [ ]:
for color in [True, False]:
    rc('axes', titlepad=5.0)
    fig, axs = plt.subplots(nrows=3, ncols=3, figsize=(8,8))
    for idx, (gericht, ax) in enumerate(zip(
        gerichte_list,
        itertools.chain.from_iterable(axs)
    )):
        df = df_decision_ref[df_decision_ref.Gericht == gericht] if gericht else df_decision_ref
        plot_linreg( 
            df['Referenzen (Binär: §/Art)'], 
            df['Token'], 
            title=gericht, 
            ax=ax,
            xlim=70,
            ylim=17500,
            show_x_label=bool(idx > 5),
            show_y_label=bool(idx % 3 == 0),
            ylabel='Tokens',
            color=color
        )
    axs[-1, -1].axis('off')
    plt.subplots_adjust(top=0.97, bottom=0.07, left=0.09, right=0.99, wspace = 0.1)
    fig.savefig(f'{data_figures_path}/makro_decisions_reference_binary_para_art_token_scatter{"" if color else "_graycolor"}.png', dpi=300)

In [ ]:
for color in [True, False]:
    rc('axes', titlepad=5.0)
    fig, axs = plt.subplots(nrows=3, ncols=3, figsize=(8,8))
    for idx, (gericht, ax) in enumerate(zip(
        gerichte_list,
        itertools.chain.from_iterable(axs)
    )):
        df = df_decision_ref[df_decision_ref.Gericht == gericht] if gericht else df_decision_ref
        plot_linreg( 
            df['Referenzen (Binär: Gesetz)'], 
            df['Token'], 
            title=gericht, 
            ax=ax,
            xlim=20,
            ylim=17500,
            xjitter=True,
            show_x_label=bool(idx > 5),
            show_y_label=bool(idx % 3 == 0),
            ylabel='Tokens',
            color=color
        )
    axs[-1, -1].axis('off')
    plt.subplots_adjust(top=0.97, bottom=0.07, left=0.09, right=0.99, wspace = 0.1)
    fig.savefig(f'{data_figures_path}/makro_decisions_reference_binary_para_art_token_scatter{"" if color else "_graycolor"}.png', dpi=300)

In [ ]:
for color in [True, False]:
    rc('axes', titlepad=5.0)
    fig, axs = plt.subplots(ncols=3, figsize=(9,3))
    df = df_decision_ref[df_decision_ref.Gericht == 'BVerfG']
    plot_linreg( 
        df['Referenzen (Gewichtet)'], 
        df['Token'], 
        title='BVerfG', 
        ax=axs[0],
        xlim=600,
        ylim=17500,
        ylabel='Tokens',
        color=color
    )
    plot_linreg( 
        df['Referenzen (Binär: §/Art)'], 
        df['Token'], 
        title='BVerfG', 
        ax=axs[1],
        xlim=55,
        ylim=17500,
        xjitter=True,
        show_y_label=False,
        ylabel='Tokens',
        color=color
    )
    plot_linreg( 
        df['Referenzen (Binär: Gesetz)'], 
        df['Token'], 
        title='BVerfG', 
        ax=axs[2],
        xlim=20,
        ylim=17500,
        xjitter=True,
        show_y_label=False,
        ylabel='Tokens',
        color=color
    )
    plt.subplots_adjust(top=0.91, bottom=0.18, left=0.09, right=0.99, wspace = 0.1)
    fig.savefig(f'{data_figures_path}/makro_decisions_reference_token_scatter_BVerfG{"" if color else "_graycolor"}.png', dpi=300)